<a href="https://colab.research.google.com/github/harideegee/dlgenai_iitmbs/blob/main/week2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
!pip install -q opendatasets

In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import opendatasets as od
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (classification_report, confusion_matrix,
                             roc_auc_score, roc_curve, precision_recall_curve,
                             f1_score, precision_score, recall_score)
from sklearn.utils.class_weight import compute_class_weight

np.random.seed(75)
torch.manual_seed(75)
if torch.cuda.is_available():
  torch.cuda.manual_seed(75)

plt.style.use('default')
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 12
sns.set_palette("husl")
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f"Environment Setup:")
print(f"Device: {device}")
print(f"PyTorch Version: {torch.__version__}")
print(f"Pandas Version: {pd.__version__}")
print(f"NumPy Version: {np.__version__}")

if torch.cuda.is_available():
  print(f"GPU: {torch.cuda.get_device_name(0)}")
  print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

Environment Setup:
Device: cuda
PyTorch Version: 2.11.0+cu128
Pandas Version: 2.2.2
NumPy Version: 2.0.2
GPU: Tesla T4
GPU Memory: 14.6 GB


In [5]:
dataset_url = 'https://www.kaggle.com/datasets/blastchar/telco-customer-churn'
od.download(dataset_url)

Please provide your Kaggle credentials to download this dataset. Learn more: http://bit.ly/kaggle-creds
Your Kaggle username: haroldking
Your Kaggle Key: ··········
Dataset URL: https://www.kaggle.com/datasets/blastchar/telco-customer-churn


100%|██████████| 172k/172k [00:00<00:00, 76.7MB/s]

In [6]:
df = pd.read_csv('./telco-customer-churn/WA_Fn-UseC_-Telco-Customer-Churn.csv')
df.shape

(7043, 21)

In [8]:
print("Data Cleaning and Quality Assessment")
print("=" * 50)
df_cleaned = df.copy()
original_shape = df_cleaned.shape

print(f"Starting with {original_shape[0]:,} rows and {original_shape[1]} columns")
print(f"\nChecking for duplicate customers...")
duplicate_customers = df_cleaned['customerID'].duplicated().sum()
print(f"Duplicated customerIDs: {duplicate_customers}")

if duplicate_customers > 0:
  print("Removing duplicate customer records...")
  duplicate_customers = df_cleaned.drop_duplicates(subset=['customerID'])
  print(f"Removed {duplicate_customers} duplicate records")

print("\nFixing TotalCharges data type...")
print(f"Current TotalCharges data type: {df_cleaned['TotalCharges'].dtype}")

if df_cleaned['TotalCharges'].dtype == 'object':
  non_numeric_total = df_cleaned['TotalCharges'].apply(lambda x: not str(x).replace('.', '').replace(' ', '').isdigit())
  non_numeric_count = non_numeric_total.sum()
  print(f"Non-numeric TotalCharges values: {non_numeric_count}")

if non_numeric_count > 0:
  print("Converting TotalCharges to numeric...")
  df_cleaned['TotalCharges'] = pd.to_numeric(df_cleaned['TotalCharges'], errors='coerce')
  print(f"TotalCharges converted to numeric type.")

print("\nChecking for missing values...")
missing_values = df_cleaned.isnull().sum()
missing_percent = (missing_values / len(df_cleaned) * 100).round(2)

missing_summary = pd.DataFrame({
    'Column': missing_values.index,
    'Missing Values': missing_values.values,
    'Missing %': missing_percent.values
})

missing_summary = missing_summary[missing_summary['Missing Values'] > 0]

if len(missing_summary) > 0:
  print("Missing Values found:")
  print(missing_summary.to_string(index=False))

  if 'TotalCharges' in missing_summary['Column'].values:
    total_missing_charges = df_cleaned['TotalCharges'].isnull().sum()
    print(f"\nHandling {total_missing_charges} missing TotalCharges values...")

    missing_tenure = df_cleaned[df_cleaned['TotalCharges'].isnull()]['tenure'].describe()
    print(f"Tenure stats for missing TotalCharges customers:")
    print(f"Mean tenure: {missing_tenure['mean']:.1f} months")

    if missing_tenure['mean'] < 3:
      print("Missing TotalCharges likely represent new customers")
      print("Filling missing TotalCharges with MonthlyCharges (first month)")
      df_cleaned['TotalCharges'] = df_cleaned['TotalCharges'].fillna(df_cleaned['TotalCharges'])
    else:
      print("Filling missing TotalCharges with median value")
      df_cleaned['TotalCharges'] = df_cleaned['TotalCharges'].fillna(df_cleaned['TotalCharges'].median())
    print("Handled missing TotalCharges values")
else:
  print("No missing values found")

Data Cleaning and Quality Assessment
Starting with 7,043 rows and 21 columns

Checking for duplicate customers...
Duplicated customerIDs: 0

Fixing TotalCharges data type...
Current TotalCharges data type: object
Non-numeric TotalCharges values: 11
Converting TotalCharges to numeric...
TotalCharges converted to numeric type.

Checking for missing values...
Missing Values found:
      Column  Missing Values  Missing %
TotalCharges              11       0.16

Handling 11 missing TotalCharges values...
Tenure sats for missing TotalCharges customers:
Mean tenure: 0.0 months
Missing TotalCharges likely represent new customers
Filling missing TotalCharges with MonthlyCharges (first month)
Handled missing TotalCharges values
